# Correlation with LLM-measured 3E

In [1]:
import os
import pandas as pd

# LLM Judge evaluating 3E

In [2]:
EEE_PROMPT = """
Evaluate the Doctor D's communication skills according to the 3E communication skills framework. Based on the provided dialogue, give a score solely based on Doctor D's spoken words and ignore all statements by Patient P.

Empower
1. The participant elicited the patient’s major concerns within the first 5 minutes of the conversation [1-5]
2. The participant asked for permission to share information about prognosis [1-5]
3. The participant asked how much information the patient would like concerning prognosis [1-5]
4. The participant checked the patient’s prognostic understanding by asking them to state what they understood, using a teach-back approach [1-5]
5. The participant actively encouraged the patient to ask questions using facilitative questions/statements (e.g., What questions do you have? At this point many patients have questions… etc.) [1-5]
6. The participant helped the SP make a plan regarding with whom, and when, to convey prognostic information to family members [1-5]
7. The participant gave the SP many opportunities to talk [1-5]
8. Overall, I thought the participant was empowering [1-10]

be Explicit
1. The participant described the medical situation (the cancer has spread) clearly and without euphemism or jargon [1-5]
2. The participant shared the prognosis accurately (a few months - less than one year) [1-5]
3. The participant used clear language without euphemism or jargon when sharing the prognosis [1-5]
4. The participant used difficult to understand medical jargon [1-5] (concentration check)
5. The participant did a good job NOT lecturing the patient (uninterrupted information for what seemed like a long time) [1-5]
6. Overall, I thought the participant was being explicit [1-10]

Empathize
1. The participant was generally empathetic [1-5]
2. The participant used statements of empathy [1-5]
3. The participant used silence appropriately in response to patient emotion [1-5]
4. The participant validated the SP emotional responses [1-5]
5. Overall, I thought the participant aligned with the patient’s emotional state [1-10]

Overall
1. The unseen qualities, hard to say feelings about this participant made me feel like they were an outstanding communicator [1-10]

# Output format
Return exactly this JSON dictionary format. Do not say anything else

```
{
    "Empower": [1, 2, 3, 4, 1, 2, 3, 8],
    "be Explicit": [1, 2, 3, 4, 1, 6],
    "Empathize": [1, 2, 3, 4, 5],
    "Overall": [3]
}
```

## Input dialogue
"""

In [6]:
import os, getpass
from keys import *
if not os.getenv("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")

In [7]:
from tinyagent import *

In [8]:
QUEST_MAX_POINTS_EMPOWER = 45
QUEST_MAX_POINTS_EXPLICIT = 35
QUEST_MAX_POINTS_EMPATHY = 30
QUEST_MAX_POINTS_ALL = 120

def compute_scores(json_scores):
  empower = sum(json_scores['Empower'])/QUEST_MAX_POINTS_EMPOWER
  explicit = sum(json_scores['be Explicit'])/QUEST_MAX_POINTS_EXPLICIT
  empathy = sum(json_scores['Empathize'])/QUEST_MAX_POINTS_EMPATHY
  flattened_scores = [item for sublist in list(json_scores.values()) for item in sublist]
  overall = sum(flattened_scores)/QUEST_MAX_POINTS_ALL

  return {
      'LLM_SUM_EMPOWER': empower,
      'LLM_SUM_EXPLICIT': explicit,
      'LLM_SUM_EMPATHY': empathy,
      'LLM_SUM_ALL': overall
  }

In [9]:
def get_total_score(dialogue, model="gpt-5-mini", reasoning_effort="low"):
  tiny_agent = TinyAgent(model)
  tiny_agent.set_reasoning_effort(reasoning_effort)
  tiny_agent.add_instruction(EEE_PROMPT)
  tiny_agent.add_data(dialogue)
  scores = tiny_agent.call_json()
  if scores:
    return compute_scores(scores)
  else:
    return None


In [10]:
dialogue = """..."""

get_total_score(dialogue)

{'LLM_SUM_EMPOWER': 0.17777777777777778,
 'LLM_SUM_EXPLICIT': 0.17142857142857143,
 'LLM_SUM_EMPATHY': 0.16666666666666666,
 'LLM_SUM_ALL': 0.16666666666666666}

# Evaluating SOPHIE 1.0 data

In [15]:
llm_3E = pd.read_csv('data/3E_by_llm2.csv')
llm_3E = llm_3E.dropna()
llm_3E = llm_3E.drop(columns=['empower', 'explicit', 'empathy'])
llm_3E

,filename,participant,control/sophie,pre/post,word_count,content
0,p19_post_w_speaker.txt,p19,control,post,1023.0,"Patient: Go.\nDoctor: Hi, Lois. How are you to..."
1,p23_post_w_speaker.txt,p23,control,post,901.0,"Doctor: All right, go for it. Hello, Jill. How..."
2,p23_pre_w_speaker.txt,p23,control,pre,745.0,"Doctor: It. What should happen here? Like, sho..."
3,p24_post_w_speaker.txt,p24,control,post,1361.0,Doctor: The last. Hi. How are you feeling?\nPa...
4,p24_pre_w_speaker.txt,p24,control,pre,1068.0,"Doctor: Hi, how are you doing?\nPatient: I'm o..."
...,...,...,...,...,...,...
96,p53_pre_w_speaker.txt,p53,sophie,pre,1416.0,"Patient: Go.\nDoctor: Hi, Joe.\nPatient: Hi.\n..."
97,p57_post_w_speaker.txt,p57,sophie,post,1532.0,Doctor: Is there. Wait for this one? Is there ...
98,p57_pre_w_speaker.txt,p57,sophie,pre,1504.0,"Patient: Go.\nDoctor: Hi, Lois. I received you..."
99,p7_pre_w_speaker.txt,p7,sophie,pre,1495.0,"Doctor: Thank you. Hello, John.\nPatient: Hey,..."


In [16]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

model = "gpt-5.4"
reasoning_effort = "low"

def process_row(text):
    return get_total_score(text, model, reasoning_effort)  # your API call

texts = llm_3E['content'].tolist()

num_workers = 100  # good starting point for API calls

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    results = list(executor.map(process_row, texts))

results_df = pd.DataFrame(results)
llm_3E = pd.concat([llm_3E, results_df], axis=1)

In [17]:
llm_3E.head()

,filename,participant,control/sophie,pre/post,word_count,content,LLM_SUM_EMPOWER,LLM_SUM_EXPLICIT,LLM_SUM_EMPATHY,LLM_SUM_ALL
0,p19_post_w_speaker.txt,p19,control,post,1023.0,"Patient: Go.\nDoctor: Hi, Lois. How are you to...",0.400000,0.714286,0.566667,0.541667
1,p23_post_w_speaker.txt,p23,control,post,901.0,"Doctor: All right, go for it. Hello, Jill. How...",0.355556,0.571429,0.333333,0.400000
2,p23_pre_w_speaker.txt,p23,control,pre,745.0,"Doctor: It. What should happen here? Like, sho...",0.200000,0.285714,0.166667,0.208333
3,p24_post_w_speaker.txt,p24,control,post,1361.0,Doctor: The last. Hi. How are you feeling?\nPa...,0.511111,0.485714,0.633333,0.533333
4,p24_pre_w_speaker.txt,p24,control,pre,1068.0,"Doctor: Hi, how are you doing?\nPatient: I'm o...",0.288889,0.571429,0.466667,0.408333


In [18]:
llm_3E.to_csv(f'data/sophie_1.0_3E_{model}_{reasoning_effort}.csv', index=False)

In [19]:
llm_3E = pd.read_csv(f'data/sophie_1.0_3E_{model}_{reasoning_effort}.csv')
llm_3E

,filename,participant,control/sophie,pre/post,word_count,content,LLM_SUM_EMPOWER,LLM_SUM_EXPLICIT,LLM_SUM_EMPATHY,LLM_SUM_ALL
0,p19_post_w_speaker.txt,p19,control,post,1023.0,"Patient: Go.\nDoctor: Hi, Lois. How are you to...",0.400000,0.714286,0.566667,0.541667
1,p23_post_w_speaker.txt,p23,control,post,901.0,"Doctor: All right, go for it. Hello, Jill. How...",0.355556,0.571429,0.333333,0.400000
2,p23_pre_w_speaker.txt,p23,control,pre,745.0,"Doctor: It. What should happen here? Like, sho...",0.200000,0.285714,0.166667,0.208333
3,p24_post_w_speaker.txt,p24,control,post,1361.0,Doctor: The last. Hi. How are you feeling?\nPa...,0.511111,0.485714,0.633333,0.533333
4,p24_pre_w_speaker.txt,p24,control,pre,1068.0,"Doctor: Hi, how are you doing?\nPatient: I'm o...",0.288889,0.571429,0.466667,0.408333
...,...,...,...,...,...,...,...,...,...,...
96,p53_pre_w_speaker.txt,p53,sophie,pre,1416.0,"Patient: Go.\nDoctor: Hi, Joe.\nPatient: Hi.\n...",0.355556,0.742857,0.700000,0.558333
97,p57_post_w_speaker.txt,p57,sophie,post,1532.0,Doctor: Is there. Wait for this one? Is there ...,0.288889,0.428571,0.433333,0.358333
98,p57_pre_w_speaker.txt,p57,sophie,pre,1504.0,"Patient: Go.\nDoctor: Hi, Lois. I received you...",0.466667,0.628571,0.566667,0.533333
99,p7_pre_w_speaker.txt,p7,sophie,pre,1495.0,"Doctor: Thank you. Hello, John.\nPatient: Hey,...",0.466667,0.485714,0.600000,0.483333


In [20]:
llm_3E.columns

Index(['filename', 'participant', 'control/sophie', 'pre/post', 'word_count',
       'content', 'LLM_SUM_EMPOWER', 'LLM_SUM_EXPLICIT', 'LLM_SUM_EMPATHY',
       'LLM_SUM_ALL'],
      dtype='object')

# Evaluating SOPHIE 2.0 data

In [11]:
import pandas as pd
# Load the first CSV file
feedback = pd.read_csv('data/admin_meeting_feedback.csv')

llm_ranking = feedback.rename(columns={
    "id": "feedback_id",
    "meeting_transcript": "transcript"
})[["feedback_id", "transcript"]].copy()

llm_ranking

,feedback_id,transcript
0,1,"Patient: Hey Doctor, thank you for taking the ..."
1,2,"Patient: Hey Doctor, thank you for taking the ..."
2,3,"Patient: Hey Doctor, thank you for taking the ..."
3,4,"Patient: Hey Doctor, thank you for taking the ..."
4,5,"Patient: Hey Doctor, thank you for taking the ..."
...,...,...
213,214,"Patient: Hey Doctor, thank you for taking the ..."
214,215,"Patient: Hey Doctor, thank you for taking the ..."
215,216,"Patient: Hey Doctor, thank you for taking the ..."
216,217,"Patient: Hey Doctor, thank you for taking the ..."


In [17]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

model = "gpt-5.4"
reasoning_effort = "low"
filename = f'data/llm_ranking_{model}_{reasoning_effort}.csv'

def process_row(text):
    return get_total_score(text, model, reasoning_effort)  # your API call

if os.path.exists(filename):
    saved = pd.read_csv(filename)
    done_ids = set(saved["feedback_id"])
else:
    saved = pd.DataFrame()
    done_ids = set()

todo = llm_ranking[~llm_ranking["feedback_id"].isin(done_ids)].copy()

print(f"Already done: {len(done_ids)}")
print(f"Remaining: {len(todo)}")

Already done: 218
Remaining: 0


In [13]:
if len(todo) > 0:
    with ThreadPoolExecutor(max_workers=100) as executor:
        results = list(executor.map(process_row, todo["transcript"]))

    results_df = pd.DataFrame(results)
    new_rows = pd.concat([todo.reset_index(drop=True), results_df], axis=1)

    llm_ranking = pd.concat([saved, new_rows], ignore_index=True)
else:
    llm_ranking = saved

llm_ranking.to_csv(filename, index=False)
llm_ranking.head()

,feedback_id,transcript,LLM_SUM_EMPOWER,LLM_SUM_EXPLICIT,LLM_SUM_EMPATHY,LLM_SUM_ALL
0,1,"Patient: Hey Doctor, thank you for taking the ...",0.311111,0.285714,0.300000,0.291667
1,2,"Patient: Hey Doctor, thank you for taking the ...",0.244444,0.428571,0.266667,0.291667
2,3,"Patient: Hey Doctor, thank you for taking the ...",0.200000,0.371429,0.233333,0.250000
3,4,"Patient: Hey Doctor, thank you for taking the ...",0.311111,0.400000,0.166667,0.291667
4,5,"Patient: Hey Doctor, thank you for taking the ...",0.177778,0.285714,0.166667,0.200000


In [14]:
llm_ranking.to_csv(filename, index=False)

In [15]:
llm_ranking.columns

Index(['feedback_id', 'transcript', 'LLM_SUM_EMPOWER', 'LLM_SUM_EXPLICIT',
       'LLM_SUM_EMPATHY', 'LLM_SUM_ALL'],
      dtype='str')

In [16]:
llm_ranking

,feedback_id,transcript,LLM_SUM_EMPOWER,LLM_SUM_EXPLICIT,LLM_SUM_EMPATHY,LLM_SUM_ALL
0,1,"Patient: Hey Doctor, thank you for taking the ...",0.311111,0.285714,0.300000,0.291667
1,2,"Patient: Hey Doctor, thank you for taking the ...",0.244444,0.428571,0.266667,0.291667
2,3,"Patient: Hey Doctor, thank you for taking the ...",0.200000,0.371429,0.233333,0.250000
3,4,"Patient: Hey Doctor, thank you for taking the ...",0.311111,0.400000,0.166667,0.291667
4,5,"Patient: Hey Doctor, thank you for taking the ...",0.177778,0.285714,0.166667,0.200000
...,...,...,...,...,...,...
213,214,"Patient: Hey Doctor, thank you for taking the ...",0.488889,0.428571,0.400000,0.450000
214,215,"Patient: Hey Doctor, thank you for taking the ...",0.200000,0.314286,0.333333,0.266667
215,216,"Patient: Hey Doctor, thank you for taking the ...",0.266667,0.371429,0.400000,0.325000
216,217,"Patient: Hey Doctor, thank you for taking the ...",0.288889,0.314286,0.166667,0.250000
